In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import os
import warnings
warnings.filterwarnings("ignore")

print("=== LSTM-AE Threshold Tuning ===\n")

# Load the trained model
model = load_model("cache/fast_lstm_ae_fixed.keras")
print("Model loaded successfully!")

# Load test data
DATA_DIR = "/Users/alex/Downloads/esa-adb-challenge (1)"
test = pd.read_parquet(f"{DATA_DIR}/test.parquet")
CHANNELS = [f"channel_{i}" for i in range(41, 47)]
WINDOW_SIZE = 40

test_scaled = test[CHANNELS].values.astype(np.float32)

# Windowing
def make_windows(data, window_size):
    n = len(data)
    return np.array([data[i:i+window_size] for i in range(0, n - window_size + 1)])

X_test = make_windows(test_scaled, WINDOW_SIZE)
print(f"Test windows shape: {X_test.shape}")

# Get reconstruction errors
def get_reconstruction_error(model, windows, batch_size=512):
    pred = model.predict(windows, batch_size=batch_size, verbose=1)
    return np.mean(np.abs(windows - pred), axis=(1, 2))

print("Computing reconstruction errors...")
test_errors = get_reconstruction_error(model, X_test)

# Map window errors to timestep scores (using max)
scores = np.zeros(len(test_scaled))
for i, err in enumerate(test_errors):
    start = i
    end = min(start + WINDOW_SIZE, len(scores))
    scores[start:end] = np.maximum(scores[start:end], err)

print(f"\nScore statistics:")
print(f"Max score   : {scores.max():.5f}")
print(f"Mean score  : {scores.mean():.5f}")
print(f"95th percentile : {np.percentile(scores, 95):.5f}")
print(f"98th percentile : {np.percentile(scores, 98):.5f}\n")

# Try multiple thresholds
thresholds = [0.26, 0.24, 0.22, 0.20, 0.18, 0.16, 0.14, 0.12]

for thresh in thresholds:
    predictions = (scores > thresh).astype(int)
    
    # Light post-processing
    from scipy.ndimage import binary_closing
    predictions = binary_closing(predictions, structure=np.ones(5)).astype(int)
    
    anomaly_count = predictions.sum()
    anomaly_rate = predictions.mean() * 100
    
    print(f"Threshold = {thresh:.3f} → Anomalies: {anomaly_count:,} ({anomaly_rate:.3f}%)")
    
    # Save submission
    sub = pd.DataFrame({
        'id': test['id'].iloc[WINDOW_SIZE-1:].reset_index(drop=True),
        'anomaly_flag': predictions[WINDOW_SIZE-1:]
    })
    sub.to_parquet(f"{DATA_DIR}/submission_lstm_thresh_{thresh:.2f}.parquet", index=False)

print("\n✅ All submissions saved. Now check the anomaly rates above.")
print("Choose the one with 1% to 5% anomaly rate and upload it to Kaggle.")

=== LSTM-AE Threshold Tuning ===

Model loaded successfully!
Test windows shape: (521241, 40, 6)
Computing reconstruction errors...
1019/1019 ━━━━━━━━━━━━━━━━━━━━ 98s 94ms/step

Score statistics:
Max score   : 0.10877
Mean score  : 0.10273
95th percentile : 0.10300
98th percentile : 0.10308

Threshold = 0.260 → Anomalies: 0 (0.000%)
Threshold = 0.240 → Anomalies: 0 (0.000%)
Threshold = 0.220 → Anomalies: 0 (0.000%)
Threshold = 0.200 → Anomalies: 0 (0.000%)
Threshold = 0.180 → Anomalies: 0 (0.000%)
Threshold = 0.160 → Anomalies: 0 (0.000%)
Threshold = 0.140 → Anomalies: 0 (0.000%)
Threshold = 0.120 → Anomalies: 0 (0.000%)

✅ All submissions saved. Now check the anomaly rates above.
Choose the one with 1% to 5% anomaly rate and upload it to Kaggle.
